In [1]:
import numpy as np
import pandas as pd

# ------------------------------ settings ------------------------------
INPUT_CSV = "aqi_processed_data.csv"            # your original data
OUTPUT_CSV = "aqi_with_lag_features.csv"
TARGET = "pm2_5_ugm3"

# ------------------------------ load data -----------------------------
df = pd.read_csv(INPUT_CSV, parse_dates=["datetime"])
df = df.sort_values("datetime").drop_duplicates("datetime").set_index("datetime")

# make sure every hour exists (lags only make sense on a continuous hourly index)
full_index = pd.date_range(df.index.min(), df.index.max(), freq="h")
df = df.reindex(full_index)
df.index.name = "datetime"
df = df.interpolate(limit=3)           # fill only short gaps

# ------------------------------ lag features --------------------------
# value of the column N hours ago, e.g. pm2_5_ugm3_lag24 = PM2.5 24 hours ago
for col in [TARGET, "pm10_ugm3"]:
    for lag in [1, 2, 3, 6, 12, 24, 48, 72, 168]:
        df[f"{col}_lag{lag}"] = df[col].shift(lag)

for col in ["no2_ugm3", "co_ugm3", "so2_ugm3", "o3_ugm3"]:
    for lag in [3, 24]:
        df[f"{col}_lag{lag}"] = df[col].shift(lag)

# ------------------------------ rolling features ----------------------
# summaries of the recent past (they include the current hour, which is known)
pm = df[TARGET]
for window in [6, 24, 72, 168]:
    df[f"pm_roll{window}_mean"] = pm.rolling(window).mean()
df["pm_roll24_max"] = pm.rolling(24).max()
df["pm_roll72_max"] = pm.rolling(72).max()
df["pm_roll24_std"] = pm.rolling(24).std()
df["pm_trend_24"] = pm - pm.shift(24)

df["rain_6h"] = df["precipitation_mm"].rolling(6).sum()
df["rain_24h"] = df["precipitation_mm"].rolling(24).sum()
df["wind_roll24_mean"] = df["wind_speed_10m_kmh"].rolling(24).mean()
df["stagnation_24h"] = df["wind_stagnation"].rolling(24).sum()

# ------------------------------ wind components -----------------------
# wind direction is circular (359 degrees is next to 1 degree), so split it in two
rad = np.deg2rad(df["wind_dir_10m"])
df["wind_u"] = -df["wind_speed_10m_kmh"] * np.sin(rad)
df["wind_v"] = -df["wind_speed_10m_kmh"] * np.cos(rad)

# ------------------------------ optional target -----------------------
# PM2.5 value 24 hours in the FUTURE (this is what you predict, not an input feature)
ADD_TARGET = True
if ADD_TARGET:
    df["target_pm25_next24h"] = df[TARGET].shift(-24)

# ------------------------------ clean and save ------------------------
df = df.dropna()                       # removes the first ~168 rows (no history yet) and the last 24 (no future target)
df = df.reset_index()
df.to_csv(OUTPUT_CSV, index=False)

print("Saved:", OUTPUT_CSV)
print("Shape:", df.shape)
print("Date range:", df["datetime"].min(), "->", df["datetime"].max())

Saved: aqi_with_lag_features.csv
Shape: (28848, 66)
Date range: 2022-08-12 00:00:00 -> 2025-11-25 23:00:00
